# 14 — Cause and effect: did the funding make a difference? (Book 2, Chapter 14)

Builds the supported and matched comparison groups for EDCTP and the Gates Foundation, fetches papers per year for each researcher, and runs the balance table, the difference-in-differences estimate and the year-by-year view. Run on 17 November 2026.

In [1]:
# Settings
import os, subprocess, sys, pandas as pd
RUN_FETCH = False   # True fetches papers per year for about 1,700 researchers (one request each); needs OPENALEX_API_KEY
os.environ.setdefault("RAW_DIR", "../data/raw")
assert not RUN_FETCH or os.environ.get("OPENALEX_API_KEY"), "Set OPENALEX_API_KEY as an environment variable, never in a file"
pd.set_option("display.width", 200)

## 1. Researchers based in sub-Saharan Africa, and those on papers naming each funder

In [2]:
subprocess.run([sys.executable, "../src/14a_build_groups.py"], check=True)

authorships 78873 SSA authorships 11540 SSA authors 5513
EDCTP SSA authors on funded papers 657 first year by period: {Interval(1990, 2004, closed='right'): 0, Interval(2004, 2008, closed='right'): 0, Interval(2008, 2012, closed='right'): 37, Interval(2012, 2016, closed='right'): 111, Interval(2016, 2020, closed='right'): 49, Interval(2020, 2026, closed='right'): 460}
Gates SSA authors on funded papers 930 first year by period: {Interval(1990, 2004, closed='right'): 0, Interval(2004, 2008, closed='right'): 54, Interval(2008, 2012, closed='right'): 98, Interval(2012, 2016, closed='right'): 110, Interval(2016, 2020, closed='right'): 94, Interval(2020, 2026, closed='right'): 574}


CompletedProcess(args=['/usr/local/bin/python3', '../src/14a_build_groups.py'], returncode=0)

## 2. Matching, and papers per year for every researcher

In [3]:
subprocess.run([sys.executable, "../src/14b_match_fetch.py"] + (["fetch"] if RUN_FETCH else []), check=True)

{('EDCTP', 'comparison'): 1056, ('EDCTP', 'supported'): 352, ('Gates', 'comparison'): 1257, ('Gates', 'supported'): 419} unique authors 1672


CompletedProcess(args=['/usr/local/bin/python3', '../src/14b_match_fetch.py'], returncode=0)

## 3. Balance, difference-in-differences and the year-by-year view

In [4]:
subprocess.run([sys.executable, "../src/14c_did.py"], check=True, capture_output=True)
display(pd.read_csv("../data/checks/funding_balance.csv").round(2))
display(pd.read_csv("../data/checks/funding_did.csv").query("outcome == 'papers'").round(2))
pd.read_csv("../data/checks/funding_event_study.csv").round(2)

,funder,role,pre_papers,flagged,people,first_corpus_year
0,EDCTP,comparison,3.16,0.09,594,2013.85
1,EDCTP,supported,3.87,0.03,352,2014.74
2,Gates,comparison,3.08,0.08,692,2014.02
3,Gates,supported,3.96,0.08,419,2015.04


,funder,sample,outcome,pairs,pre_supported,pre_comparison,post_supported,post_comparison,naive_post_ratio,did,lo,hi
0,EDCTP,all,papers,352,3.87,3.16,5.04,3.46,1.46,0.86,0.44,1.29
2,EDCTP,no flagged,papers,264,4.18,3.56,5.41,3.90,1.39,0.90,0.43,1.38
4,Gates,all,papers,419,3.96,3.08,4.99,3.39,1.47,0.73,0.40,1.08
6,Gates,no flagged,papers,304,4.42,3.37,5.64,3.73,1.51,0.86,0.43,1.29


,funder,e,supported,comparison,gap_vs_e-1
0,EDCTP,-3,3.23,2.61,-0.60
1,EDCTP,-2,3.51,3.20,-0.91
2,EDCTP,-1,4.88,3.66,0.00
3,EDCTP,0,6.01,3.26,1.53
4,EDCTP,1,5.16,3.52,0.42
5,EDCTP,2,4.91,3.37,0.32
6,EDCTP,3,5.03,3.48,0.33
7,Gates,-3,3.23,2.49,-0.02
8,Gates,-2,4.15,3.04,0.34
9,Gates,-1,4.49,3.72,0.00


**Check.** The start year is excluded from the estimate; the version without flagged profiles should be close to the full one.

In [5]:
r = pd.read_csv("../data/checks/funding_did.csv").query("outcome == 'papers'").set_index(["funder", "sample"]).did
for f in ["EDCTP", "Gates"]:
    print(f, round(r[(f, "all")], 2), round(r[(f, "no flagged")], 2))
    assert abs(r[(f, "all")] - r[(f, "no flagged")]) < 0.3

EDCTP 0.86 0.9
Gates 0.73 0.86


## 4. Sensitivity: first funded paper with 20 authors or fewer

On large trial papers, a funder named may mean one researcher among dozens. Keeping only researchers whose first funded paper had a small team tests whether the result depends on them.

In [6]:
pd.read_csv("../data/checks/funding_did_small_teams.csv").round(2)

,funder,start paper,pairs,did,lo,hi
0,EDCTP,20 authors or fewer,189,1.31,0.69,1.95
1,Gates,20 authors or fewer,221,1.24,0.78,1.73
